# Numerical ECG Classification — 53 labels with PCA

In [1]:
import numpy as np
import pandas as pd
import json
import joblib
import re

from pathlib import Path
from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score
)

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, TensorDataset


/usr/lib/python3/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset

In [2]:
RAW_DIR = Path("data/ptb-xl-plus/median_beats/12sl")

# Existing ECG IDs, patient folds and labels only
INDEX_DIR = Path("data/processed/12sl_median_images_final")

ROOT_DIR = Path("data/processed/12sl_raw_models")

NUM_LABELS = 53

EXPERIMENT_ROOT = ROOT_DIR / "pca_models"

DATA_DIR = EXPERIMENT_ROOT / f"top_{NUM_LABELS}"

DATA_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    INDEX_DIR / "training_index_with_folds.csv"
)

with open(INDEX_DIR / "selected_labels.json") as f:

    selected_labels = json.load(f)


In [3]:
train_df = df[df["strat_fold"].between(1, 8)].copy()
val_df = df[df["strat_fold"] == 9].copy()
test_df = df[df["strat_fold"] == 10].copy()

assert df["ecg_id"].is_unique
assert df[["patient_id", "strat_fold"]].notna().all().all()

assert set(train_df.patient_id).isdisjoint(val_df.patient_id)
assert set(train_df.patient_id).isdisjoint(test_df.patient_id)
assert set(val_df.patient_id).isdisjoint(test_df.patient_id)

assert len(selected_labels) == 53

assert (len(train_df), len(val_df), len(test_df)) == (
    16608, 2148, 2158
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))
print("Selected labels:", len(selected_labels))


Training: 16608
Validation: 2148
Testing: 2158
Selected labels: 53


In [4]:
with open(DATA_DIR / "selected_labels.json", "w") as f:

    json.dump(selected_labels, f, indent=4)

distribution = pd.DataFrame({
    "Train": train_df[selected_labels].sum(),
    "Validation": val_df[selected_labels].sum()
})

assert (distribution["Train"] > 0).all()
assert (distribution["Train"] < len(train_df)).all()
assert (distribution["Validation"] > 0).all()
assert (distribution["Validation"] < len(val_df)).all()

display(distribution)

distribution.to_csv(DATA_DIR / "label_distribution.csv")


,Train,Validation
label_NSR,9641,1253
label_AB,8770,1069
label_NML,4590,635
label_SBRAD,2594,317
label_SRTH,1939,255
label_ABR,1931,256
label_IMI;AU,1714,196
label_LAD3,1495,187
label_BORDE,1317,188
label_AFIB,1071,129


In [5]:
LEAD_ORDER = [
    "I", "II", "III",
    "aVR", "aVL", "aVF",
    "V1", "V2", "V3",
    "V4", "V5", "V6"
]

raw_headers = {}

for path in RAW_DIR.rglob("*_medians.hea"):

    ecg_id = int(path.stem.split("_")[0])

    assert ecg_id not in raw_headers

    raw_headers[ecg_id] = path

assert set(df["ecg_id"]).issubset(raw_headers)

print("Original ECG records:", len(raw_headers))


Original ECG records: 20914


In [6]:
def load_signal(ecg_id):
    header_path = raw_headers[int(ecg_id)]
    lines = [
        line.strip()
        for line in header_path.read_text().splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]
    record = lines[0].split()
    assert int(record[1]) == 12
    assert float(record[2]) == 500
    assert int(record[3]) == 600
    assert len(lines) >= 13

    raw = np.fromfile(
        header_path.with_suffix(".dat"),
        dtype="<i4"
    )
    assert raw.size == 600 * 12
    raw = raw.reshape(600, 12)
    assert not np.any(raw == np.iinfo(np.int32).min)

    signal = np.empty((600, 12), dtype=np.float32)
    lead_names = []
    for i, line in enumerate(lines[1:13]):
        fields = line.split()
        assert int(fields[1]) == 32
        match = re.fullmatch(
            r"([0-9.eE+-]+)(?:\(([-+0-9.eE]+)\))?/(.+)",
            fields[2]
        )
        assert match is not None
        gain = float(match.group(1))
        baseline = (
            float(match.group(2))
            if match.group(2) is not None
            else float(fields[4])
        )
        assert np.isfinite(gain) and gain > 0
        assert np.isfinite(baseline)
        assert int(raw[0, i]) == int(fields[5])
        checksum = int(raw[:, i].sum(dtype=np.int64)) & 0xFFFF
        assert checksum == (int(fields[6]) & 0xFFFF)
        signal[:, i] = (
            (raw[:, i].astype(np.float64) - baseline)
            / gain / 1000.0
        )
        lead_names.append(fields[8])
    assert len(lead_names) == 12
    assert set(lead_names) == set(LEAD_ORDER)
    order = [lead_names.index(lead) for lead in LEAD_ORDER]
    signal = signal[:, order].T.copy()
    assert signal.shape == (12, 600)
    assert np.isfinite(signal).all()
    return signal


In [7]:
sample_signal = load_signal(
    train_df.iloc[0]["ecg_id"]
)

print("Signal shape:", sample_signal.shape)
print("Number of leads:", sample_signal.shape[0])
print("Samples per lead:", sample_signal.shape[1])
print("Minimum amplitude:", sample_signal.min())
print("Maximum amplitude:", sample_signal.max())

print(sample_signal[:, :10])


Signal shape: (12, 600)
Number of leads: 12
Samples per lead: 600
Minimum amplitude: -1.41
Maximum amplitude: 0.629
[[-0.024 -0.039 -0.048 -0.039 -0.039 -0.034 -0.034 -0.034 -0.034 -0.034]
 [ 0.019  0.019  0.019  0.014  0.014  0.014  0.019  0.019  0.019  0.019]
 [ 0.043  0.058  0.068  0.053  0.053  0.048  0.053  0.053  0.053  0.053]
 [ 0.     0.009  0.014  0.009  0.009  0.009  0.004  0.004  0.004  0.004]
 [-0.034 -0.048 -0.058 -0.043 -0.043 -0.039 -0.043 -0.043 -0.043 -0.043]
 [ 0.034  0.039  0.043  0.034  0.034  0.029  0.039  0.039  0.039  0.039]
 [-0.004 -0.004 -0.004 -0.004 -0.004 -0.004 -0.004 -0.004 -0.004 -0.004]
 [-0.043 -0.043 -0.043 -0.043 -0.043 -0.043 -0.043 -0.039 -0.039 -0.039]
 [ 0.004  0.004  0.004  0.004  0.004  0.004  0.004  0.004  0.004  0.004]
 [ 0.014  0.014  0.014  0.014  0.014  0.014  0.014  0.014  0.014  0.014]
 [ 0.024  0.024  0.024  0.024  0.024  0.024  0.024  0.024  0.024  0.024]
 [ 0.014  0.014  0.014  0.014  0.014  0.014  0.014  0.014  0.014  0.014]]


In [8]:
def prepare_signals(dataframe, split_name):

    X = np.empty(
        (len(dataframe), 12, 600),
        dtype=np.float32
    )

    for i, ecg_id in enumerate(
        tqdm(dataframe["ecg_id"],
             desc=f"Loading {split_name} signals")
    ):

        X[i] = load_signal(ecg_id)

    return X


In [9]:
SIGNAL_DIR = ROOT_DIR / "raw_signals"

SIGNAL_DIR.mkdir(exist_ok=True)

def get_signals(dataframe, split_name):

    data_path = SIGNAL_DIR / f"X_{split_name}.npy"
    ids_path = SIGNAL_DIR / f"{split_name}_ids.npy"

    ids = dataframe["ecg_id"].to_numpy(dtype=np.int64)

    if data_path.exists() and ids_path.exists():

        saved_ids = np.load(ids_path)

        if np.array_equal(saved_ids, ids):

            X = np.load(data_path, mmap_mode="r")

            if X.shape == (len(dataframe), 12, 600):

                return X

    X = prepare_signals(dataframe, split_name)

    np.save(data_path, X)
    np.save(ids_path, ids)

    return np.load(data_path, mmap_mode="r")


In [10]:
X_train_signal = get_signals(train_df, "train")
X_val_signal = get_signals(val_df, "val")

y_train = train_df[selected_labels].to_numpy(dtype=np.uint8)
y_val = val_df[selected_labels].to_numpy(dtype=np.uint8)

print("Training:", X_train_signal.shape, y_train.shape)
print("Validation:", X_val_signal.shape, y_val.shape)

np.save(DATA_DIR / "y_train.npy", y_train)
np.save(DATA_DIR / "y_val.npy", y_val)


Training: (16608, 12, 600) (16608, 53)
Validation: (2148, 12, 600) (2148, 53)


In [11]:
signal_mean = X_train_signal.mean()
signal_std = X_train_signal.std()

assert signal_std > 0

print("Mean:", signal_mean)
print("Standard deviation:", signal_std)

np.save(
    DATA_DIR / "normalization.npy",
    np.array([signal_mean, signal_std])
)


Mean: 0.028459946
Standard deviation: 0.1760315


## Baseline models

In [12]:
# Flatten the original numerical ECG signal: 12 x 600 = 7200.
# Use the same training-only normalization as the 1D CNN.

X_train = (
    (X_train_signal - signal_mean) / signal_std
).reshape(len(train_df), -1)

X_val = (
    (X_val_signal - signal_mean) / signal_std
).reshape(len(val_df), -1)

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)

assert X_train.shape[1] == 7200
assert X_val.shape[1] == 7200


Training features: (16608, 7200)
Validation features: (2148, 7200)


### PCA for Logistic Regression, Random Forest and SVM

In [13]:
from sklearn.decomposition import PCA

PCA_DIR = ROOT_DIR / "pca_features_256"

PCA_DIR.mkdir(exist_ok=True)


In [14]:
train_ids = train_df["ecg_id"].to_numpy()
val_ids = val_df["ecg_id"].to_numpy()

normalization = np.array([signal_mean, signal_std])

pca_files = [
    "pca.pkl",
    "X_train_pca.npy",
    "X_val_pca.npy",
    "train_ids.npy",
    "val_ids.npy",
    "normalization.npy"
]

use_saved_pca = all(
    (PCA_DIR / file).exists()
    for file in pca_files
)

if use_saved_pca:

    use_saved_pca = (
        np.array_equal(
            np.load(PCA_DIR / "train_ids.npy"), train_ids
        )
        and np.array_equal(
            np.load(PCA_DIR / "val_ids.npy"), val_ids
        )
        and np.array_equal(
            np.load(PCA_DIR / "normalization.npy"), normalization
        )
    )


In [15]:
if use_saved_pca:

    pca = joblib.load(PCA_DIR / "pca.pkl")

    X_train_pca = np.load(PCA_DIR / "X_train_pca.npy")
    X_val_pca = np.load(PCA_DIR / "X_val_pca.npy")

else:

    pca = PCA(
        n_components=256,
        svd_solver="randomized",
        random_state=42
    )

    X_train_pca = pca.fit_transform(X_train)
    X_val_pca = pca.transform(X_val)

    joblib.dump(pca, PCA_DIR / "pca.pkl")

    np.save(PCA_DIR / "X_train_pca.npy", X_train_pca)
    np.save(PCA_DIR / "X_val_pca.npy", X_val_pca)

    np.save(PCA_DIR / "train_ids.npy", train_ids)
    np.save(PCA_DIR / "val_ids.npy", val_ids)
    np.save(PCA_DIR / "normalization.npy", normalization)


In [16]:
assert X_train_pca.shape == (len(train_df), 256)
assert X_val_pca.shape == (len(val_df), 256)

print("Training PCA features:", X_train_pca.shape)
print("Validation PCA features:", X_val_pca.shape)

print("Explained variance:",
      pca.explained_variance_ratio_.sum())


Training PCA features: (16608, 256)
Validation PCA features: (2148, 256)
Explained variance: 0.99721247


### Logistic Regression

In [17]:
lr_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=1000,
        solver="liblinear",
        random_state=42
    ),
    n_jobs=2,
    verbose=10
)


In [18]:
lr_model.fit(X_train_pca, y_train)


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:   10.0s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:   20.5s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:   38.5s
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:  1.0min
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:  1.5min
[Parallel(n_jobs=2)]: Done  28 tasks      | elapsed:  1.8min
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed:  2.4min
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:  2.9min
[Parallel(n_jobs=2)]: Done  53 out of  53 | elapsed:  3.4min finished


,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...r='liblinear')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",2
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",10
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](53,)","[ 0, 1, 2,...,50,51,52]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear'), ...]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,53
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,256
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42


In [19]:
print("Logistic Regression training completed.")

joblib.dump(
    lr_model,
    DATA_DIR / "logistic_regression_pca.pkl"
)


Logistic Regression training completed.


['data/processed/12sl_raw_models/pca_models/top_53/logistic_regression_pca.pkl']

In [20]:
lr_val_probs = lr_model.predict_proba(X_val_pca)

lr_val_preds = (lr_val_probs >= 0.5).astype(int)

print("Macro AUROC:",
      roc_auc_score(y_val, lr_val_probs, average="macro"))

print("Macro AUPRC:",
      average_precision_score(y_val, lr_val_probs,
                              average="macro"))

print("Macro F1:",
      f1_score(y_val, lr_val_preds,
               average="macro", zero_division=0))

print("Micro F1:",
      f1_score(y_val, lr_val_preds,
               average="micro", zero_division=0))


Macro AUROC: 0.8384323301999428
Macro AUPRC: 0.2867825209521135
Macro F1: 0.2526901522123338
Micro F1: 0.5743249147950712


### Random Forest

In [21]:
rf_model = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=18,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),
    n_jobs=1,
    verbose=10
)


In [22]:
rf_model.fit(X_train_pca, y_train)


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    3.1s
[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:   11.2s
[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:   20.4s
[Parallel(n_jobs=1)]: Done  12 tasks      | elapsed:   37.7s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:   52.7s
[Parallel(n_jobs=1)]: Done  24 tasks      | elapsed:  1.3min
[Parallel(n_jobs=1)]: Done  31 tasks      | elapsed:  1.7min
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:  2.2min
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:  2.7min
[Parallel(n_jobs=1)]: Done  53 out of  53 | elapsed:  2.9min finished


,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",RandomForestC...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",10
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](53,)","[ 0, 1, 2,...,50,51,52]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[RandomForestC...ndom_state=42), RandomForestC...ndom_state=42), RandomForestC...ndom_state=42), RandomForestC...ndom_state=42), ...]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,53
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,256
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",18


In [23]:
print("Random Forest training completed.")

joblib.dump(
    rf_model,
    DATA_DIR / "random_forest_pca.pkl"
)


Random Forest training completed.


['data/processed/12sl_raw_models/pca_models/top_53/random_forest_pca.pkl']

In [24]:
rf_val_probs = rf_model.predict_proba(X_val_pca)

rf_val_preds = (rf_val_probs >= 0.5).astype(int)

print("Macro AUROC:",
      roc_auc_score(y_val, rf_val_probs, average="macro"))

print("Macro AUPRC:",
      average_precision_score(y_val, rf_val_probs,
                              average="macro"))

print("Macro F1:",
      f1_score(y_val, rf_val_preds,
               average="macro", zero_division=0))

print("Micro F1:",
      f1_score(y_val, rf_val_preds,
               average="micro", zero_division=0))


Macro AUROC: 0.8215731420597067
Macro AUPRC: 0.21867669040788584
Macro F1: 0.06756459972208115
Micro F1: 0.49468639299854134


### SVM

In [25]:
svm_model = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        dual=False,
        max_iter=5000,
        random_state=42
    ),
    n_jobs=2,
    verbose=10
)


In [26]:
svm_model.fit(X_train_pca, y_train)


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:    6.5s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:   19.7s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:   33.8s
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:   45.5s
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:   59.9s
/home/aarish/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
[Parallel(n_jobs=2)]: Done  28 tasks      | elapsed:  1.5min
/home/aarish/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed:  1.9min
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:  2.5min
/home/aarish/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: Conve

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LinearSVC(dua...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",2
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",10
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](53,)","[ 0, 1, 2,...,50,51,52]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LinearSVC(dua...ndom_state=42), LinearSVC(dua...ndom_state=42), LinearSVC(dua...ndom_state=42), LinearSVC(dua...ndom_state=42), ...]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,53
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,256
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",False


In [27]:
print("SVM training completed.")

joblib.dump(svm_model, DATA_DIR / "linear_svm_pca.pkl")


SVM training completed.


['data/processed/12sl_raw_models/pca_models/top_53/linear_svm_pca.pkl']

In [28]:
svm_val_scores = svm_model.decision_function(X_val_pca)

svm_val_preds = (svm_val_scores >= 0).astype(int)

print("Macro AUROC:",
      roc_auc_score(y_val, svm_val_scores, average="macro"))

print("Macro AUPRC:",
      average_precision_score(y_val, svm_val_scores,
                              average="macro"))

print("Macro F1:",
      f1_score(y_val, svm_val_preds,
               average="macro", zero_division=0))

print("Micro F1:",
      f1_score(y_val, svm_val_preds,
               average="micro", zero_division=0))


Macro AUROC: 0.8343876673025404
Macro AUPRC: 0.2842279021327584
Macro F1: 0.22761303416377604
Micro F1: 0.5709090909090909


In [29]:
baseline_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "SVM"],

    "Macro AUROC": [
        roc_auc_score(y_val, lr_val_probs, average="macro"),
        roc_auc_score(y_val, rf_val_probs, average="macro"),
        roc_auc_score(y_val, svm_val_scores, average="macro")
    ],

    "Macro AUPRC": [
        average_precision_score(y_val, lr_val_probs, average="macro"),
        average_precision_score(y_val, rf_val_probs, average="macro"),
        average_precision_score(y_val, svm_val_scores, average="macro")
    ],

    "Macro F1": [
        f1_score(y_val, lr_val_preds, average="macro", zero_division=0),
        f1_score(y_val, rf_val_preds, average="macro", zero_division=0),
        f1_score(y_val, svm_val_preds, average="macro", zero_division=0)
    ],

    "Micro F1": [
        f1_score(y_val, lr_val_preds, average="micro", zero_division=0),
        f1_score(y_val, rf_val_preds, average="micro", zero_division=0),
        f1_score(y_val, svm_val_preds, average="micro", zero_division=0)
    ]
})

display(baseline_results.round(4))

baseline_results.to_csv(
    DATA_DIR / "baseline_validation_results.csv",
    index=False
)


,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
0,Logistic Regression,0.8384,0.2868,0.2527,0.5743
1,Random Forest,0.8216,0.2187,0.0676,0.4947
2,SVM,0.8344,0.2842,0.2276,0.5709


## Neural network training

In [30]:
torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 0.0003
PATIENCE = 5


Device: cuda


In [31]:
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    running_loss = 0.0

    for signals, targets in tqdm(loader, desc="Training", leave=False):

        signals = signals.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        # Forward propagation
        logits = model(signals)

        # Calculate loss
        loss = criterion(logits, targets)

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        running_loss += loss.item() * signals.size(0)

    return running_loss / len(loader.dataset)


In [32]:
def evaluate_model(model, loader, criterion):

    model.eval()

    running_loss = 0.0

    all_targets = []
    all_probs = []

    with torch.no_grad():

        for signals, targets in tqdm(
            loader, desc="Evaluating", leave=False
        ):

            signals = signals.to(device)
            targets = targets.to(device)

            logits = model(signals)

            loss = criterion(logits, targets)

            probs = torch.sigmoid(logits)

            running_loss += loss.item() * signals.size(0)

            all_targets.append(targets.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    y_true = np.concatenate(all_targets, axis=0)
    y_probs = np.concatenate(all_probs, axis=0)

    y_preds = (y_probs >= 0.5).astype(int)

    metrics = {
        "loss": running_loss / len(loader.dataset),

        "macro_auroc": roc_auc_score(
            y_true, y_probs, average="macro"
        ),

        "macro_auprc": average_precision_score(
            y_true, y_probs, average="macro"
        ),

        "macro_f1": f1_score(
            y_true, y_preds,
            average="macro", zero_division=0
        ),

        "micro_f1": f1_score(
            y_true, y_preds,
            average="micro", zero_division=0
        )
    }

    return metrics, y_true, y_probs


### MLP

In [33]:
train_dataset_mlp = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float()
)

val_dataset_mlp = TensorDataset(
    torch.from_numpy(X_val).float(),
    torch.from_numpy(y_val).float()
)

train_loader_mlp = DataLoader(
    train_dataset_mlp,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader_mlp = DataLoader(
    val_dataset_mlp,
    batch_size=BATCH_SIZE,
    shuffle=False
)


/tmp/ipykernel_90267/779102908.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:208.)
  torch.from_numpy(y_train).float()


In [34]:
signals, labels = next(iter(train_loader_mlp))

print("Signal batch shape:", signals.shape)
print("Label batch shape:", labels.shape)


Signal batch shape: torch.Size([16, 7200])
Label batch shape: torch.Size([16, 53])


In [35]:
class ECGMLP(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(7200, 256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        return self.classifier(x)


In [36]:
torch.manual_seed(42)
np.random.seed(42)

model_mlp = ECGMLP(
    num_classes=len(selected_labels)
).to(device)

print(model_mlp)

print("Trainable parameters:",
      sum(p.numel() for p in model_mlp.parameters()
          if p.requires_grad))


ECGMLP(
  (classifier): Sequential(
    (0): Linear(in_features=7200, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.4, inplace=False)
    (6): Linear(in_features=128, out_features=53, bias=True)
  )
)
Trainable parameters: 1883189


In [37]:
criterion_mlp = nn.BCEWithLogitsLoss()

optimizer_mlp = torch.optim.AdamW(
    model_mlp.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler_mlp = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_mlp,
    mode="max",
    factor=0.5,
    patience=2
)

BEST_MODEL_PATH_MLP = DATA_DIR / "mlp_best.pt"


In [38]:
history_mlp = {
    "train_loss": [],
    "val_loss": [],
    "macro_auroc": [],
    "macro_auprc": [],
    "macro_f1": [],
    "micro_f1": []
}

best_auprc_mlp = -np.inf
best_epoch_mlp = 0
epochs_without_improvement_mlp = 0


In [39]:
for epoch in range(EPOCHS):

    train_loss_mlp = train_one_epoch(
        model_mlp,
        train_loader_mlp,
        optimizer_mlp,
        criterion_mlp
    )

    val_metrics_mlp, _, _ = evaluate_model(
        model_mlp,
        val_loader_mlp,
        criterion_mlp
    )

    history_mlp["train_loss"].append(train_loss_mlp)
    history_mlp["val_loss"].append(val_metrics_mlp["loss"])
    history_mlp["macro_auroc"].append(val_metrics_mlp["macro_auroc"])
    history_mlp["macro_auprc"].append(val_metrics_mlp["macro_auprc"])
    history_mlp["macro_f1"].append(val_metrics_mlp["macro_f1"])
    history_mlp["micro_f1"].append(val_metrics_mlp["micro_f1"])

    scheduler_mlp.step(val_metrics_mlp["macro_auprc"])

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss_mlp:.4f} | "
        f"Val Loss: {val_metrics_mlp['loss']:.4f} | "
        f"AUROC: {val_metrics_mlp['macro_auroc']:.4f} | "
        f"AUPRC: {val_metrics_mlp['macro_auprc']:.4f} | "
        f"Macro F1: {val_metrics_mlp['macro_f1']:.4f}"
    )

    if val_metrics_mlp["macro_auprc"] > best_auprc_mlp:

        best_auprc_mlp = val_metrics_mlp["macro_auprc"]
        best_epoch_mlp = epoch + 1

        torch.save(
            model_mlp.state_dict(),
            BEST_MODEL_PATH_MLP
        )

        epochs_without_improvement_mlp = 0

        print("Best model saved.")

    else:

        epochs_without_improvement_mlp += 1

    if epochs_without_improvement_mlp >= PATIENCE:

        print("Early stopping triggered.")
        break


Epoch [01/30] | Train Loss: 0.1571 | Val Loss: 0.1198 | AUROC: 0.8314 | AUPRC: 0.2756 | Macro F1: 0.1705
Best model saved.


Epoch [02/30] | Train Loss: 0.1265 | Val Loss: 0.1112 | AUROC: 0.8593 | AUPRC: 0.3135 | Macro F1: 0.1785
Best model saved.


Epoch [03/30] | Train Loss: 0.1198 | Val Loss: 0.1062 | AUROC: 0.8743 | AUPRC: 0.3420 | Macro F1: 0.2109
Best model saved.


Epoch [04/30] | Train Loss: 0.1155 | Val Loss: 0.1047 | AUROC: 0.8803 | AUPRC: 0.3586 | Macro F1: 0.2333
Best model saved.


Epoch [05/30] | Train Loss: 0.1123 | Val Loss: 0.1039 | AUROC: 0.8835 | AUPRC: 0.3655 | Macro F1: 0.2441
Best model saved.


Epoch [06/30] | Train Loss: 0.1099 | Val Loss: 0.1030 | AUROC: 0.8885 | AUPRC: 0.3724 | Macro F1: 0.2514
Best model saved.


... (20 outputs left out)

Epoch [27/30] | Train Loss: 0.0927 | Val Loss: 0.0950 | AUROC: 0.9037 | AUPRC: 0.4144 | Macro F1: 0.2873


Epoch [28/30] | Train Loss: 0.0922 | Val Loss: 0.0942 | AUROC: 0.9035 | AUPRC: 0.4173 | Macro F1: 0.2862


Epoch [29/30] | Train Loss: 0.0889 | Val Loss: 0.0937 | AUROC: 0.9048 | AUPRC: 0.4184 | Macro F1: 0.3020


Epoch [30/30] | Train Loss: 0.0877 | Val Loss: 0.0934 | AUROC: 0.9050 | AUPRC: 0.4270 | Macro F1: 0.3029
Best model saved.


In [40]:
print("\nTraining completed.")
print("Best epoch:", best_epoch_mlp)
print("Best validation Macro AUPRC:", best_auprc_mlp)

history_mlp_df = pd.DataFrame(history_mlp)

history_mlp_df.to_csv(
    DATA_DIR / "mlp_training_history.csv",
    index=False
)



Training completed.
Best epoch: 30
Best validation Macro AUPRC: 0.42704484486007876


In [41]:
model_mlp.load_state_dict(
    torch.load(
        BEST_MODEL_PATH_MLP,
        map_location=device
    )
)

mlp_metrics, mlp_val_targets, mlp_val_probs = evaluate_model(
    model_mlp,
    val_loader_mlp,
    criterion_mlp
)

mlp_val_preds = (mlp_val_probs >= 0.5).astype(int)

assert np.array_equal(mlp_val_targets, y_val)

print("Best MLP Validation Results")

for metric, value in mlp_metrics.items():

    print(f"{metric}: {value:.4f}")


Best MLP Validation Results
loss: 0.0934
macro_auroc: 0.9050
macro_auprc: 0.4270
macro_f1: 0.3029
micro_f1: 0.6806


In [42]:
mlp_results = pd.DataFrame([{
    "Model": "MLP",
    "Macro AUROC": mlp_metrics["macro_auroc"],
    "Macro AUPRC": mlp_metrics["macro_auprc"],
    "Macro F1": mlp_metrics["macro_f1"],
    "Micro F1": mlp_metrics["micro_f1"]
}])

baseline_results = pd.concat(
    [baseline_results, mlp_results],
    ignore_index=True
)

display(baseline_results.round(4))

baseline_results.to_csv(
    DATA_DIR / "baseline_validation_results.csv",
    index=False
)

np.save(
    DATA_DIR / "mlp_val_probs.npy",
    mlp_val_probs
)


,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
0,Logistic Regression,0.8384,0.2868,0.2527,0.5743
1,Random Forest,0.8216,0.2187,0.0676,0.4947
2,SVM,0.8344,0.2842,0.2276,0.5709
3,MLP,0.9050,0.4270,0.3029,0.6806


In [43]:
# Release flattened arrays before training the 1D CNN.

del train_loader_mlp, val_loader_mlp
del train_dataset_mlp, val_dataset_mlp
del model_mlp, optimizer_mlp, scheduler_mlp
del lr_model, rf_model, svm_model
del X_train, X_val
del X_train_pca, X_val_pca, pca


## 1D CNN

In [44]:
class ECGSignalDataset(Dataset):

    def __init__(self, signals, dataframe, labels):

        self.signals = signals

        self.targets = dataframe[labels].to_numpy(
            dtype=np.float32, copy=True
        )

    def __len__(self):

        return len(self.signals)

    def __getitem__(self, idx):

        signal = self.signals[idx]

        signal = (signal - signal_mean) / signal_std

        signal = torch.from_numpy(signal).float()

        target = torch.from_numpy(self.targets[idx])

        return signal, target


In [45]:
train_dataset_1d = ECGSignalDataset(
    X_train_signal, train_df, selected_labels
)

val_dataset_1d = ECGSignalDataset(
    X_val_signal, val_df, selected_labels
)

train_loader_1d = DataLoader(
    train_dataset_1d,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader_1d = DataLoader(
    val_dataset_1d,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [46]:
signals, labels = next(iter(train_loader_1d))

print("Signal batch shape:", signals.shape)
print("Label batch shape:", labels.shape)
print("Training batches:", len(train_loader_1d))


Signal batch shape: torch.Size([16, 12, 600])
Label batch shape: torch.Size([16, 53])
Training batches: 1038


In [47]:
class ECGCNN1D(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        def conv_block(in_channels, out_channels):

            return nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=5,
                    padding=2,
                    bias=False
                ),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool1d(2)
            )

        self.features = nn.Sequential(
            conv_block(12, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256)
        )

        self.pool = nn.AdaptiveAvgPool1d(8)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)

        return x


In [48]:
torch.manual_seed(42)
np.random.seed(42)

model_1d = ECGCNN1D(
    num_classes=len(selected_labels)
).to(device)

print(model_1d)

print("Trainable parameters:",
      sum(p.numel() for p in model_1d.parameters()
          if p.requires_grad))


ECGCNN1D(
  (features): Sequential(
    (0): Sequential(
      (0): Conv1d(12, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, c

In [49]:
criterion_1d = nn.BCEWithLogitsLoss()

optimizer_1d = torch.optim.AdamW(
    model_1d.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler_1d = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_1d,
    mode="max",
    factor=0.5,
    patience=2
)

BEST_MODEL_PATH_1D = DATA_DIR / "cnn_1d_best.pt"


In [50]:
history_1d = {
    "train_loss": [],
    "val_loss": [],
    "macro_auroc": [],
    "macro_auprc": [],
    "macro_f1": [],
    "micro_f1": []
}

best_auprc_1d = -np.inf
best_epoch_1d = 0
epochs_without_improvement_1d = 0


In [51]:
for epoch in range(EPOCHS):

    train_loss_1d = train_one_epoch(
        model_1d,
        train_loader_1d,
        optimizer_1d,
        criterion_1d
    )

    val_metrics_1d, _, _ = evaluate_model(
        model_1d,
        val_loader_1d,
        criterion_1d
    )

    history_1d["train_loss"].append(train_loss_1d)
    history_1d["val_loss"].append(val_metrics_1d["loss"])
    history_1d["macro_auroc"].append(val_metrics_1d["macro_auroc"])
    history_1d["macro_auprc"].append(val_metrics_1d["macro_auprc"])
    history_1d["macro_f1"].append(val_metrics_1d["macro_f1"])
    history_1d["micro_f1"].append(val_metrics_1d["micro_f1"])

    scheduler_1d.step(val_metrics_1d["macro_auprc"])

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss_1d:.4f} | "
        f"Val Loss: {val_metrics_1d['loss']:.4f} | "
        f"AUROC: {val_metrics_1d['macro_auroc']:.4f} | "
        f"AUPRC: {val_metrics_1d['macro_auprc']:.4f} | "
        f"Macro F1: {val_metrics_1d['macro_f1']:.4f}"
    )

    if val_metrics_1d["macro_auprc"] > best_auprc_1d:

        best_auprc_1d = val_metrics_1d["macro_auprc"]
        best_epoch_1d = epoch + 1

        torch.save(
            model_1d.state_dict(),
            BEST_MODEL_PATH_1D
        )

        epochs_without_improvement_1d = 0

        print("Best model saved.")

    else:

        epochs_without_improvement_1d += 1

    if epochs_without_improvement_1d >= PATIENCE:

        print("Early stopping triggered.")
        break


Epoch [01/30] | Train Loss: 0.1373 | Val Loss: 0.1073 | AUROC: 0.8783 | AUPRC: 0.3455 | Macro F1: 0.1458
Best model saved.


Epoch [02/30] | Train Loss: 0.1084 | Val Loss: 0.0969 | AUROC: 0.8999 | AUPRC: 0.3939 | Macro F1: 0.2230
Best model saved.


Epoch [03/30] | Train Loss: 0.1009 | Val Loss: 0.0931 | AUROC: 0.9043 | AUPRC: 0.4235 | Macro F1: 0.2588
Best model saved.


Epoch [04/30] | Train Loss: 0.0962 | Val Loss: 0.0894 | AUROC: 0.9130 | AUPRC: 0.4458 | Macro F1: 0.2705
Best model saved.


Epoch [05/30] | Train Loss: 0.0929 | Val Loss: 0.0906 | AUROC: 0.9156 | AUPRC: 0.4447 | Macro F1: 0.2836


Epoch [06/30] | Train Loss: 0.0901 | Val Loss: 0.0863 | AUROC: 0.9198 | AUPRC: 0.4661 | Macro F1: 0.3072
Best model saved.


... (20 outputs left out)

Epoch [27/30] | Train Loss: 0.0681 | Val Loss: 0.0808 | AUROC: 0.9335 | AUPRC: 0.5187 | Macro F1: 0.3931


Epoch [28/30] | Train Loss: 0.0663 | Val Loss: 0.0819 | AUROC: 0.9347 | AUPRC: 0.5151 | Macro F1: 0.3763


Epoch [29/30] | Train Loss: 0.0664 | Val Loss: 0.0809 | AUROC: 0.9352 | AUPRC: 0.5205 | Macro F1: 0.3919


Epoch [30/30] | Train Loss: 0.0620 | Val Loss: 0.0811 | AUROC: 0.9359 | AUPRC: 0.5259 | Macro F1: 0.3892


In [52]:
print("\nTraining completed.")
print("Best epoch:", best_epoch_1d)
print("Best validation Macro AUPRC:", best_auprc_1d)

history_1d_df = pd.DataFrame(history_1d)

history_1d_df.to_csv(
    DATA_DIR / "cnn_1d_training_history.csv",
    index=False
)



Training completed.
Best epoch: 26
Best validation Macro AUPRC: 0.5294232807844721


In [53]:
model_1d.load_state_dict(
    torch.load(
        BEST_MODEL_PATH_1D,
        map_location=device
    )
)

cnn_1d_metrics, cnn_1d_val_targets, cnn_1d_val_probs = evaluate_model(
    model_1d,
    val_loader_1d,
    criterion_1d
)

cnn_1d_val_preds = (cnn_1d_val_probs >= 0.5).astype(int)

assert np.array_equal(cnn_1d_val_targets, y_val)

print("Best 1D CNN Validation Results")

for metric, value in cnn_1d_metrics.items():

    print(f"{metric}: {value:.4f}")


Best 1D CNN Validation Results
loss: 0.0804
macro_auroc: 0.9357
macro_auprc: 0.5294
macro_f1: 0.3863
micro_f1: 0.7280


In [54]:
cnn_1d_results = pd.DataFrame([{
    "Model": "1D CNN",
    "Macro AUROC": cnn_1d_metrics["macro_auroc"],
    "Macro AUPRC": cnn_1d_metrics["macro_auprc"],
    "Macro F1": cnn_1d_metrics["macro_f1"],
    "Micro F1": cnn_1d_metrics["micro_f1"]
}])

display(cnn_1d_results.round(4))

np.save(
    DATA_DIR / "cnn_1d_val_probs.npy",
    cnn_1d_val_probs
)


,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
0,1D CNN,0.9357,0.5294,0.3863,0.728


## Model comparison

In [55]:
final_val_results = pd.concat(
    [baseline_results, cnn_1d_results],
    ignore_index=True
)

final_val_results.insert(0, "Number of labels", NUM_LABELS)

final_val_results["Input"] = np.where(
    final_val_results["Model"].isin([
        "Logistic Regression", "Random Forest", "SVM"
    ]),
    "PCA (256 features)",
    "Original numerical ECG"
)

display(final_val_results.round(4))

final_val_results.to_csv(
    DATA_DIR / "validation_model_comparison.csv",
    index=False
)


,Number of labels,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1,Input
0,53,Logistic Regression,0.8384,0.2868,0.2527,0.5743,PCA (256 features)
1,53,Random Forest,0.8216,0.2187,0.0676,0.4947,PCA (256 features)
2,53,SVM,0.8344,0.2842,0.2276,0.5709,PCA (256 features)
3,53,MLP,0.9050,0.4270,0.3029,0.6806,Original numerical ECG
4,53,1D CNN,0.9357,0.5294,0.3863,0.7280,Original numerical ECG


## Per-label evaluation

In [56]:
model_scores = {
    "Logistic Regression": (lr_val_probs, 0.5),
    "Random Forest": (rf_val_probs, 0.5),
    "SVM": (svm_val_scores, 0.0),
    "MLP": (mlp_val_probs, 0.5),
    "1D CNN": (cnn_1d_val_probs, 0.5)
}

per_label_results = []

for name, (scores, threshold) in model_scores.items():

    assert scores.shape == y_val.shape

    preds = (scores >= threshold).astype(int)

    for i, label in enumerate(selected_labels):

        per_label_results.append({
            "Number of labels": NUM_LABELS,
            "Model": name,
            "Label": label,
            "Validation positives": int(y_val[:, i].sum()),
            "AUROC": roc_auc_score(
                y_val[:, i], scores[:, i]
            ),
            "AUPRC": average_precision_score(
                y_val[:, i], scores[:, i]
            ),
            "F1": f1_score(
                y_val[:, i], preds[:, i], zero_division=0
            )
        })

    file_name = name.lower().replace(" ", "_")

    np.savez(
        DATA_DIR / f"{file_name}_validation.npz",
        scores=scores,
        targets=y_val,
        labels=np.asarray(selected_labels),
        ecg_ids=val_df["ecg_id"].to_numpy(dtype=np.int64),
        threshold=threshold
    )

per_label_results = pd.DataFrame(per_label_results)

display(per_label_results.round(4))

per_label_results.to_csv(
    DATA_DIR / "per_label_validation_results.csv",
    index=False
)


,Number of labels,Model,Label,Validation positives,AUROC,AUPRC,F1
0,53,Logistic Regression,label_NSR,1253,0.8547,0.8584,0.8359
1,53,Logistic Regression,label_AB,1069,0.9127,0.9284,0.8293
2,53,Logistic Regression,label_NML,635,0.8603,0.6540,0.6100
3,53,Logistic Regression,label_SBRAD,317,0.9318,0.6567,0.6296
4,53,Logistic Regression,label_SRTH,255,0.6770,0.2897,0.1656
...,...,...,...,...,...,...,...
260,53,1D CNN,label_FLUT,18,0.9419,0.5981,0.5385
261,53,1D CNN,label_IVCD,11,0.9955,0.7103,0.0000
262,53,1D CNN,label_$SABCOND,14,0.6916,0.0405,0.0000
263,53,1D CNN,label_BIFB,12,0.9881,0.4364,0.4211


## Comparison on the five common labels

In [57]:
all_label_cols = [
    col for col in df.columns
    if col.startswith("label_")
]

common_labels = (
    train_df[all_label_cols].sum()
    .sort_values(ascending=False, kind="stable")
    .head(5).index.tolist()
)

common_indices = [
    selected_labels.index(label)
    for label in common_labels
]

with open(DATA_DIR / "common_labels.json", "w") as f:

    json.dump(common_labels, f, indent=4)

np.save(
    DATA_DIR / "validation_ecg_ids.npy",
    val_df["ecg_id"].to_numpy(dtype=np.int64)
)


In [58]:
common_results = []

for name, (scores, threshold) in model_scores.items():

    targets = y_val[:, common_indices]

    scores_5 = scores[:, common_indices]

    preds = (scores_5 >= threshold).astype(int)

    common_results.append({
        "Number of labels": NUM_LABELS,
        "Model": name,

        "Macro AUROC": roc_auc_score(
            targets, scores_5, average="macro"
        ),

        "Macro AUPRC": average_precision_score(
            targets, scores_5, average="macro"
        ),

        "Macro F1": f1_score(
            targets, preds,
            average="macro", zero_division=0
        ),

        "Micro F1": f1_score(
            targets, preds,
            average="micro", zero_division=0
        )
    })

common_results = pd.DataFrame(common_results)

display(common_results.round(4))

common_results.to_csv(
    DATA_DIR / "common_top5_results.csv",
    index=False
)


,Number of labels,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
0,53,Logistic Regression,0.8473,0.6775,0.6141,0.7478
1,53,Random Forest,0.8447,0.6784,0.4723,0.7220
2,53,SVM,0.8469,0.6804,0.6000,0.7462
3,53,MLP,0.9046,0.7908,0.7172,0.8324
4,53,1D CNN,0.9261,0.8329,0.7586,0.8554


## Compare 5, 10 and 53-label experiments

In [59]:
results = []

for n in [5, 10, 53]:

    path = (
        EXPERIMENT_ROOT / f"top_{n}"
        / "validation_model_comparison.csv"
    )

    if path.exists():

        results.append(pd.read_csv(path))

comparison = pd.concat(results, ignore_index=True)

display(comparison.round(4))

comparison.to_csv(
    EXPERIMENT_ROOT / "all_label_experiments.csv",
    index=False
)


,Number of labels,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1,Input
0,5,Logistic Regression,0.8473,0.6775,0.6141,0.7478,PCA (256 features)
1,5,Random Forest,0.8447,0.6784,0.4723,0.7220,PCA (256 features)
2,5,SVM,0.8469,0.6804,0.6000,0.7462,PCA (256 features)
3,5,MLP,0.9115,0.8024,0.7253,0.8321,Original numerical ECG
4,5,1D CNN,0.9297,0.8370,0.7544,0.8476,Original numerical ECG
5,10,Logistic Regression,0.8667,0.5681,0.5031,0.6853,PCA (256 features)
6,10,Random Forest,0.8416,0.5096,0.2405,0.6288,PCA (256 features)
7,10,SVM,0.8670,0.5705,0.4610,0.6778,PCA (256 features)
8,10,MLP,0.9280,0.7380,0.6792,0.7994,Original numerical ECG
9,10,1D CNN,0.9512,0.8201,0.7535,0.8324,Original numerical ECG


In [60]:
comparison_table = comparison.pivot(
    index="Model",
    columns="Number of labels",
    values=[
        "Macro AUROC", "Macro AUPRC",
        "Macro F1", "Micro F1"
    ]
)

display(comparison_table.round(4))


Macro AUROC                 Macro AUPRC                  \
Number of labels             5       10      53          5       10      53   
Model                                                                         
1D CNN                   0.9297  0.9512  0.9357      0.8370  0.8201  0.5294   
Logistic Regression      0.8473  0.8667  0.8384      0.6775  0.5681  0.2868   
MLP                      0.9115  0.9280  0.9050      0.8024  0.7380  0.4270   
Random Forest            0.8447  0.8416  0.8216      0.6784  0.5096  0.2187   
SVM                      0.8469  0.8670  0.8344      0.6804  0.5705  0.2842   

                    Macro F1                 Micro F1                  
Number of labels          5       10      53       5       10      53  
Model                                                                  
1D CNN                0.7544  0.7535  0.3863   0.8476  0.8324  0.7280  
Logistic Regression   0.6141  0.5031  0.2527   0.7478  0.6853  0.5743  
MLP                   0.7253  0.6792  0.3029   0.8321  0.7994  0.6806  
Random Forest         0.4723  0.2405  0.0676   0.7220  0.6288  0.4947  
SVM                   0.6000  0.4610  0.2276   0.7462  0.6778  0.5709

In [61]:
common_results_list = []

for n in [5, 10, 53]:

    folder = EXPERIMENT_ROOT / f"top_{n}"
    path = folder / "common_top5_results.csv"

    if path.exists():

        with open(folder / "common_labels.json") as f:

            assert json.load(f) == common_labels

        assert np.array_equal(
            np.load(folder / "validation_ecg_ids.npy"),
            val_df["ecg_id"].to_numpy()
        )

        common_results_list.append(
            pd.read_csv(path)
        )

common_comparison = pd.concat(
    common_results_list,
    ignore_index=True
)

display(common_comparison.round(4))

common_comparison.to_csv(
    EXPERIMENT_ROOT / "common_top5_comparison.csv",
    index=False
)


,Number of labels,Model,Macro AUROC,Macro AUPRC,Macro F1,Micro F1
0,5,Logistic Regression,0.8473,0.6775,0.6141,0.7478
1,5,Random Forest,0.8447,0.6784,0.4723,0.7220
2,5,SVM,0.8469,0.6804,0.6000,0.7462
3,5,MLP,0.9115,0.8024,0.7253,0.8321
4,5,1D CNN,0.9297,0.8370,0.7544,0.8476
5,10,Logistic Regression,0.8473,0.6775,0.6141,0.7478
6,10,Random Forest,0.8447,0.6784,0.4723,0.7220
7,10,SVM,0.8469,0.6804,0.6000,0.7462
8,10,MLP,0.9114,0.8013,0.7297,0.8406
9,10,1D CNN,0.9345,0.8428,0.7728,0.8583
